# Week 4 Exercises: APIs, Data Wrangling & PostgreSQL

**Data Engineering Course · Pipeline Africa**

---

Work through these exercises in order. Each section builds on the last.

**Setup:**
```
pip install requests pandas psycopg2-binary python-dotenv
```

In [1]:
import os
import requests
import pandas as pd
import json
from pathlib import Path

DATA = Path('../data')
print('Ready.')

Ready.


---

## Section 1: REST APIs (Exercises 1–5)

### Exercise 1

Make a GET request to `https://api.open-meteo.com/v1/forecast` for Nairobi, Kenya (latitude `-1.2921`, longitude `36.8219`, timezone `Africa/Nairobi`).

Request these daily fields: `temperature_2m_max`, `temperature_2m_min`, `precipitation_sum`.

Print:
- The HTTP status code
- The timezone in the response
- The first date in the `daily.time` list

In [2]:
# Exercise 1 — your code here

url = 'https://api.open-meteo.com/v1/forecast'

params = {
    'latitude':  -1.2921,      # Lagos, Nigeria
    'longitude': 36.8219,
    'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone':  'Africa/Nairobi',
}

response = requests.get(url, params=params)
data = response.json()
print(f'Status code: {response.status_code}')   # 200 = OK
print(f'URL called:  {response.url}') 
print(f'Data: {data}')
print(f'timezone data: {data["timezone"]}')
print(f'time data: {data["daily"]["time"][0]}')

Status code: 200
URL called:  https://api.open-meteo.com/v1/forecast?latitude=-1.2921&longitude=36.8219&daily=temperature_2m_max%2Ctemperature_2m_min%2Cprecipitation_sum&timezone=Africa%2FNairobi
Data: {'latitude': -1.3005272, 'longitude': 36.824646, 'generationtime_ms': 3.5387277603149414, 'utc_offset_seconds': 10800, 'timezone': 'Africa/Nairobi', 'timezone_abbreviation': 'GMT+3', 'elevation': 1668.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_sum': 'mm'}, 'daily': {'time': ['2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-17', '2026-10-18'], 'temperature_2m_max': [28.1, 29.0, 30.1, 26.7, 27.0, 27.1, 27.9], 'temperature_2m_min': [14.1, 13.6, 15.8, 16.3, 15.0, 13.8, 15.5], 'precipitation_sum': [0.0, 0.0, 0.1, 16.1, 1.1, 0.0, 1.2]}}
timezone data: Africa/Nairobi
time data: 2026-10-12


### Exercise 2

Write a function called `get_forecast(lat, lon, tz)` that:
- Makes a GET request to Open-Meteo
- Returns the full parsed JSON dict on success
- Returns `None` if the request fails (use `try/except`)
- Passes `timeout=10` to `requests.get()`

Test it with Accra, Ghana (lat `5.6037`, lon `-0.1870`, tz `Africa/Accra`).

In [3]:
# Exercise 2 — your code here
def get_forecast(lat, lon, tz):
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'timezone':  tz,
    }
    try:
        response = requests.get(url, params=params, timeout=10)  
        response.raise_for_status()  
        data = response.json()
        return data
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        return None
get_forecast(5.6037, -0.1870, 'Africa/Accra')


{'latitude': 5.5887523,
 'longitude': -0.22406006,
 'generationtime_ms': 0.04792213439941406,
 'utc_offset_seconds': 0,
 'timezone': 'Africa/Accra',
 'timezone_abbreviation': 'GMT',
 'elevation': 56.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-10-12',
   '2026-10-13',
   '2026-10-14',
   '2026-10-15',
   '2026-10-16',
   '2026-10-17',
   '2026-10-18'],
  'temperature_2m_max': [30.5, 30.8, 30.6, 30.7, 31.8, 31.3, 30.4],
  'temperature_2m_min': [24.9, 24.3, 24.3, 24.5, 24.5, 24.8, 24.7],
  'precipitation_sum': [2.1, 0.8, 1.7, 0.7, 0.9, 0.6, 1.2]}}

### Exercise 3

What happens when you send a bad status code? Call the URL below and:
- Print the status code
- Print the JSON error message from the response

```python
bad_url = 'https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&timezone=UTC'
```

In [4]:
# Exercise 3 — your code here
bad_url = 'https://api.open-meteo.com/v1/forecast?latitude=999&longitude=999&daily=temperature_2m_max&timezone=UTC'
params = {
    'latitude':  -1.2921,
    'longitude': 36.8219,
    'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone':  'Africa/Nairobi',
}

response = requests.get(bad_url, params=params)
data = response.json()
print(f'Status code: {response.status_code}')
print(f'Data: {data}')

Status code: 400
Data: {'reason': 'Latitude must be in range of -90 to 90°. Given: 999.0.', 'error': True}


### Exercise 4

Fetch the forecast for Cairo (lat `30.0444`, lon `31.2357`, tz `Africa/Cairo`) and save the raw JSON response to `../data/cairo_forecast.json`.

Then read the file back and print the number of days in the response.

In [5]:
# Exercise 4 — your code here
response_data = get_forecast(30.0444, 31.2357, 'Africa/Cairo')
if response_data:
    out = DATA / 'cairo_forecast.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(response_data, f, indent=2)
    print(f'Saved raw response to {out}')

with open(DATA / 'cairo_forecast.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(f'Re-loaded: {len(loaded["daily"]["time"])} days')

Saved raw response to ..\data\cairo_forecast.json
Re-loaded: 7 days


### Exercise 5

The Open-Meteo API supports hourly data as well as daily. Make a request for Lagos (lat `6.5244`, lon `3.3792`) that asks for the **hourly** field `temperature_2m`. You will need to change the `daily` parameter to `hourly`.

Print how many hourly values are returned. (Hint: 7 days × 24 hours = ?)

In [6]:
# Exercise 5 — your code here
url = 'https://api.open-meteo.com/v1/forecast'

params = {
    'latitude':  6.5244,      
    'longitude': 3.3792,
    'hourly':     'temperature_2m',
    'timezone':  'Africa/Lagos',
}

response = requests.get(url, params=params)
data = response.json()
print(f'Status code: {response.status_code}')
print(f'Hourly temperatures: {len(data["hourly"]["temperature_2m"])}')

Status code: 200
Hourly temperatures: 168


---

## Section 2: Data Wrangling (Exercises 6–10)

### Exercise 6

Load `../data/sample_weather.json` and build a DataFrame from the `daily` data.

Rename the columns:
- `time` → `date`
- `temperature_2m_max` → `temp_max_c`
- `temperature_2m_min` → `temp_min_c`
- `precipitation_sum` → `rain_mm`
- `windspeed_10m_max` → `wind_max_kmh`

Print the DataFrame.

In [7]:
# Exercise 6 — your code here
try:
    with open(DATA / 'sample_weather.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)
except FileNotFoundError:
    with open(DATA / 'sample_weather.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

df = pd.DataFrame(raw['daily'])  
df = df.rename(columns={
    'time':                'date',
    'temperature_2m_max':  'temp_max_c',
    'temperature_2m_min':  'temp_min_c',
    'precipitation_sum':   'rain_mm',
    'windspeed_10m_max':   'wind_max_kmh',
})

df['date'] = pd.to_datetime(df['date'])
print(df.head())      


        date  temp_max_c  temp_min_c  rain_mm  wind_max_kmh
0 2024-01-01        33.1        24.2      0.0          14.8
1 2024-01-02        32.8        23.8      0.0          12.3
2 2024-01-03        31.5        24.1      2.4          18.6
3 2024-01-04        33.4        24.5      0.0          10.2
4 2024-01-05        34.2        25.1      0.0          11.5


### Exercise 7

Add a column called `temp_range_c` that shows the difference between `temp_max_c` and `temp_min_c` for each day.

Add a column called `is_rainy` that is `True` when `rain_mm > 0`, and `False` otherwise.

Print the updated DataFrame.

In [9]:
# Exercise 7 — your code here
df['temp_range_c']  = df['temp_max_c'] - df['temp_min_c']
df['is_rainy'] = df['rain_mm'] > 0

df.head()

,date,temp_max_c,temp_min_c,rain_mm,wind_max_kmh,temp_range_c,is_rainy
0,2024-01-01,33.1,24.2,0.0,14.8,8.9,False
1,2024-01-02,32.8,23.8,0.0,12.3,9.0,False
2,2024-01-03,31.5,24.1,2.4,18.6,7.4,True
3,2024-01-04,33.4,24.5,0.0,10.2,8.9,False
4,2024-01-05,34.2,25.1,0.0,11.5,9.1,False


### Exercise 8

Fetch the 7-day forecast for all 5 cities in `../data/cities.csv` using the `get_forecast()` function from Exercise 2.

Build a combined DataFrame. It must include a `city` column so you can tell the rows apart.

Print the shape of the combined DataFrame and the first 5 rows.

In [18]:
# Exercise 8 — your code here
import time
cities_df = pd.read_csv(DATA / 'cities.csv')
print(cities_df)

all_frames = []

for _, row in cities_df.iterrows():
    print(f'Fetching {row["city"]}...')
    df = get_forecast(row['latitude'], row['longitude'], row['timezone'])
    if df:
        df = pd.DataFrame(raw['daily']) 
        df = df.rename(columns={
                'time':                'date',
                'temperature_2m_max':  'temp_max_c',
                'temperature_2m_min':  'temp_min_c',
                'precipitation_sum':   'rain_mm',
                'windspeed_10m_max':   'wind_max_kmh',
        })
        
        # Convert the date column from string to proper datetime
        df['date'] = pd.to_datetime(df['date'])
        df['city'] = row['city']
        all_frames.append(df)
    time.sleep(0.3)   # be polite — do not hammer the server


if all_frames:
    weather = pd.concat(all_frames, ignore_index=True)
    print(f'\nCombined: {weather.shape}')
    print(weather.head())
else:
    print('No data fetched.')

           city       country  latitude  longitude             timezone
0         Lagos       Nigeria    6.5244     3.3792         Africa/Lagos
1       Nairobi         Kenya   -1.2921    36.8219       Africa/Nairobi
2         Accra         Ghana    5.6037    -0.1870         Africa/Accra
3  Johannesburg  South Africa  -26.2041    28.0473  Africa/Johannesburg
4         Cairo         Egypt   30.0444    31.2357         Africa/Cairo
Fetching Lagos...
Fetching Nairobi...
Fetching Accra...
Fetching Johannesburg...
Fetching Cairo...

Combined: (35, 6)
        date  temp_max_c  temp_min_c  rain_mm  wind_max_kmh   city
0 2024-01-01        33.1        24.2      0.0          14.8  Lagos
1 2024-01-02        32.8        23.8      0.0          12.3  Lagos
2 2024-01-03        31.5        24.1      2.4          18.6  Lagos
3 2024-01-04        33.4        24.5      0.0          10.2  Lagos
4 2024-01-05        34.2        25.1      0.0          11.5  Lagos


### Exercise 9

Using the combined DataFrame from Exercise 8:

1. Which city has the highest **average maximum temperature** this week?
2. Which city has the highest **total rainfall** this week?
3. On how many days does Cairo have wind speed above 20 km/h?

Answer each question with a short pandas expression.

In [11]:
weather[weather['city'] == 'Cairo']

,date,temp_max_c,temp_min_c,rain_mm,wind_max_kmh,city
28,2024-01-01,33.1,24.2,0.0,14.8,Cairo
29,2024-01-02,32.8,23.8,0.0,12.3,Cairo
30,2024-01-03,31.5,24.1,2.4,18.6,Cairo
31,2024-01-04,33.4,24.5,0.0,10.2,Cairo
32,2024-01-05,34.2,25.1,0.0,11.5,Cairo
33,2024-01-06,33.7,24.8,8.2,22.4,Cairo
34,2024-01-07,32.9,24.0,1.1,16.7,Cairo


In [12]:
# Exercise 9 — your code here
average_temp = weather.groupby("city")["temp_max_c"].mean()
print(average_temp)
highest_rainfall = weather.groupby("city")["rain_mm"].sum().idxmax()
print(f'City with highest rainfall: {highest_rainfall}')
wind_speed = (weather[weather["city"] == "Cairo"]["wind_max_kmh"]>20).sum()
print(f'Wind speed in Cairo exceeds 20 km/h: {wind_speed}')

city
Accra           33.085714
Cairo           33.085714
Johannesburg    33.085714
Lagos           33.085714
Nairobi         33.085714
Name: temp_max_c, dtype: float64
City with highest rainfall: Accra
Wind speed in Cairo exceeds 20 km/h: 1


### Exercise 10

Write a validation function called `check_weather_data(df)` that:
- Checks there are no null values in `temp_max_c`, `temp_min_c`
- Checks that `temp_max_c > temp_min_c` for every row
- Checks that `rain_mm >= 0` for every row
- Prints 'Validation passed' if everything is OK
- Prints a specific error message for each problem it finds

Run it on your combined DataFrame.

In [13]:
# Exercise 10 — your code here
def check_weather_data(df):
    issues = []
    if df['temp_max_c'].isnull():
        issues.append("Missing maximum temperature data.")
    
    if df['temp_min_c'].isnull():
        issues.append("Missing minimum temperature data.")
    
    if df['temp_max_c'] > df['temp_min_c']:
        issues.append("Maximum temperature is less than minimum temperature.")
    
    if df['rain_mm'] .lt(0).any():
        issues.append("Rainfall data is negative.")
    
    if issues:
            print('Validation issues found:')
            for issue in issues:
                print(f'  - {issue}')
    else:
            print('Validation passed') 

check_weather_data(weather)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

---

## Section 3: PostgreSQL (Exercises 11–15)

*These exercises require PostgreSQL to be running.*

### Exercise 11

Connect to PostgreSQL using credentials from environment variables (use `os.environ.get()` with sensible defaults).

Print the server version to confirm the connection works.

In [14]:
# Exercise 11 — your code here
import psycopg2
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'dbname':   'weather_db',
    'user':     'postgres',
    'password': 'Aishat1122',   # change this
}

try:
    conn = psycopg2.connect(**DB_CONFIG)
    print('Connected to PostgreSQL!')
    print(f'Server version: {conn.server_version}')
except psycopg2.OperationalError as e:
    print(f'Connection failed: {e}')
    print('Check that PostgreSQL is running and your credentials are correct.')

Connected to PostgreSQL!
Server version: 160014


### Exercise 12

Create a table called `city_weather_ex` with these columns:
- `id` SERIAL PRIMARY KEY
- `city` VARCHAR(100) NOT NULL
- `date` DATE NOT NULL
- `temp_max_c` NUMERIC(5,2)
- `rain_mm` NUMERIC(7,2)
- UNIQUE constraint on (city, date)

Use `CREATE TABLE IF NOT EXISTS` so it is safe to re-run.

In [15]:
# Exercise 12 — your code here
create_sql = '''
    CREATE TABLE IF NOT EXISTS city_weather_ex (
        id           SERIAL PRIMARY KEY,
        city         VARCHAR(100) NOT NULL,
        date         DATE         NOT NULL,
        temp_max_c   NUMERIC(5,2),
        rain_mm      NUMERIC(7,2),
        UNIQUE (city, date)   -- no duplicate city+date combinations
    );
'''

with conn.cursor() as cur:
    cur.execute(create_sql)
conn.commit()   # save the change

print('Table city_weather_ex created (or already exists).')

Table city_weather_ex created (or already exists).


### Exercise 13

Insert the rows from your combined DataFrame (Exercise 8) into `city_weather_ex`.

Use `cur.executemany()` with a parameterised `INSERT` statement.

After inserting, print the total row count from the database.

In [23]:
# Exercise 13 — your code here
insert_sql = '''
    INSERT INTO city_weather_ex
        (date, temp_max_c, rain_mm, city)
    VALUES
        (%s, %s, %s, %s)
    on conflict (city, date) do nothing;  -- ignore duplicates'''
records = [(str(row['date']), float(row['temp_max_c']), float(row['rain_mm']), str(row['city'])) for _, row in weather.iterrows()]
try:
    cur = conn.cursor()
    cur.executemany(insert_sql, records)
    conn.commit()
    cur.execute('SELECT COUNT(*) FROM city_weather_ex;')
    count = cur.fetchone()[0]
    print(f'Inserted {count} rows into city_weather_ex')
    cur.close()
except Exception as e:
    cur.rollback()
    print(f'Error inserting data: {e}')
       
  


Inserted 35 rows into city_weather_ex


### Exercise 14

Use `pd.read_sql()` to answer this question from the database:

*Which city had the highest maximum temperature recorded in the table?*

Write a SQL query that returns the city name, date, and `temp_max_c`, ordered by temperature descending, limited to 1 row.

In [25]:
# Exercise 14 — your code here
df_db = pd.read_sql('SELECT * FROM city_weather_ex ORDER BY temp_max_c DESC LIMIT 1;', conn)
print(f'Rows in database: {len(df_db)}')
df_db.head()

Rows in database: 1


C:\Users\DELL\AppData\Local\Temp\ipykernel_14876\1259926856.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_db = pd.read_sql('SELECT * FROM city_weather_ex ORDER BY temp_max_c DESC LIMIT 1;', conn)


,id,city,date,temp_max_c,rain_mm
0,5,Lagos,2024-01-05,34.2,0.0


### Exercise 15

Change your INSERT from Exercise 13 to an **upsert** (`INSERT ... ON CONFLICT DO UPDATE`).

Run it twice. Confirm that the row count stays the same after the second run — no duplicates.

In [26]:
# Exercise 15 — your code here
upsert_sql = '''
    INSERT INTO city_weather_ex
        (date, temp_max_c, rain_mm, city)
    VALUES
        (%s, %s, %s, %s)
    on conflict (city, date)  DO UPDATE SET
        temp_max_c   = EXCLUDED.temp_max_c,
        rain_mm      = EXCLUDED.rain_mm
        ;
'''
records = [(str(row['date']), float(row['temp_max_c']), float(row['rain_mm']), str(row['city'])) for _, row in weather.iterrows()]
try:
    cur = conn.cursor()
    cur.executemany(upsert_sql, records)
    conn.commit()
    cur.execute('SELECT COUNT(*) FROM city_weather_ex;')
    count = cur.fetchone()[0]
    print(f'Upserted {count} rows into city_weather_ex')
    cur.close()
except Exception as e:
    cur.rollback()
    print(f'Error inserting data: {e}')

Upserted 35 rows into city_weather_ex


---

## Section 4: Full Pipeline Challenge (Exercises 16–18)

### Exercise 16

Add logging to the `get_forecast()` function from Exercise 2.

Configure a logger that writes to `../logs/exercises.log` AND the console.

Log:
- `INFO` when a city fetch succeeds (include the city name and number of rows)
- `ERROR` when a fetch fails (include the city name and the error)

In [28]:
# Exercise 16 — your code here
import logging
import pathlib
import datetime
from datetime import datetime
log_dir = pathlib.Path('../logs/exercises.log')
log_dir.mkdir(exist_ok=True)
logger = logging.getLogger('weather_log')
logger.setLevel(logging.DEBUG)   # capture everything; handlers control what they show

formatter = logging.Formatter(
        fmt='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S',
    )

    # Console handler — show INFO and above
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(formatter)

    # File handler — save DEBUG and above to a file
today = datetime.now().strftime('%Y-%m-%d')
log_file = log_dir / f'{'weather_log'}_{today}.log'
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

logger.addHandler(console)
logger.addHandler(file_handler)

def get_forecast(lat, lon, tz):
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'timezone':  tz,
    }
    try:
        response = requests.get(url, params=params, timeout=10)  
        response.raise_for_status()  
        data = response.json()
        logger.info(f"Successfully fetched forecast for lat={lat}, lon={lon}, tz={tz}")
        return data
    except requests.exceptions.RequestException as e:
        print(f"An error occurred: {e}")
        logger.error(f"An error occurred while fetching forecast for lat={lat}, lon={lon}, tz={tz}: {e}")
        return None
get_forecast(5.6037, -0.1870, 'Africa/Accra')


2026-10-12 13:41:28  INFO      weather_log  Successfully fetched forecast for lat=5.6037, lon=-0.187, tz=Africa/Accra


{'latitude': 5.5887523,
 'longitude': -0.22406006,
 'generationtime_ms': 0.07367134094238281,
 'utc_offset_seconds': 0,
 'timezone': 'Africa/Accra',
 'timezone_abbreviation': 'GMT',
 'elevation': 56.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_min': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2026-10-12',
   '2026-10-13',
   '2026-10-14',
   '2026-10-15',
   '2026-10-16',
   '2026-10-17',
   '2026-10-18'],
  'temperature_2m_max': [31.6, 30.7, 30.8, 30.6, 31.5, 31.8, 30.4],
  'temperature_2m_min': [25.0, 24.4, 24.7, 24.3, 24.3, 24.7, 24.8],
  'precipitation_sum': [0.9, 0.7, 1.2, 0.5, 0.6, 1.5, 1.2]}}

### Exercise 17

Write a `config.json` file for your pipeline. It should contain:
- `api.base_url`
- `api.timeout_sec`
- `api.fields`
- `cities_file`
- `output_dir`

Write a function `load_config(path)` that reads the JSON file and returns a dict. Test it by printing the API base URL.

In [31]:
# Exercise 17 — your code here
config = {
    'api': {
        'base_url':    'https://api.open-meteo.com/v1/forecast',
        'timeout_sec': 10,
        'fields':      'temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max',
        'sleep_between_requests': 0.3
    },
    'cities_file':   'data/cities.csv',
    'output_dir':    'data/clean',
    'log_dir':       'logs',
    'db': {
        'host':   'localhost',
        'port':   5432,
        'dbname': 'weather_db',
        'user':   'postgres'
    }
}

# Note: password is NOT in the config file — it will come from an environment variable

config_path = Path('config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)
def load_config():
    with open(config_path, 'r', encoding='utf-8') as f:
        return json.load(f)
config_loaded = load_config()
print(f'API base URL: {config_loaded["api"]["base_url"]}')    
print('Created config.json:')


API base URL: https://api.open-meteo.com/v1/forecast
Created config.json:


### Exercise 18 — Full Pipeline

Build a single function `run_pipeline(config_path)` that:

1. Loads the config from `config.json`
2. Loads cities from the cities CSV
3. Fetches the 7-day forecast for each city (with a 0.3-second pause between requests)
4. Validates the data
5. Saves to `data/clean/pipeline_output.csv`
6. Logs every major step

Run it and verify the output CSV exists and has the right shape.

In [35]:
# Exercise 18 — your code here
def run_pipeline(config_path):
    config = load_config(config_path)
    cities_df = pd.read_csv(DATA / 'cities.csv')
print(cities_df)

all_frames = []

for _, row in cities_df.iterrows():
    print(f'Fetching {row["city"]}...')
    logger.info(f'Fetching {row["city"]}...')
    df = get_forecast(row['latitude'], row['longitude'], row['timezone'])
    if df:
        df = pd.DataFrame(raw['daily']) 
        df = df.rename(columns={
                'time':                'date',
                'temperature_2m_max':  'temp_max_c',
                'temperature_2m_min':  'temp_min_c',
                'precipitation_sum':   'rain_mm',
                'windspeed_10m_max':   'wind_max_kmh',
        })
        
        # Convert the date column from string to proper datetime
        df['date'] = pd.to_datetime(df['date'])
        df['city'] = row['city']
        all_frames.append(df)
        logger.info(f'Successfully fetched data for {row["city"]}')
    time.sleep(0.3)   # be polite — do not hammer the server


if all_frames:
    weather = pd.concat(all_frames, ignore_index=True)
    output_file = '../data/clean/pipeline_output.csv'
    weather.to_csv(output_file, index=False)
    print(f'\nCombined: {weather.shape}')
    print(weather.head())
else:
    print('No data fetched.')
    
  

2026-10-12 14:19:13  INFO      weather_log  Fetching Lagos...


           city       country  latitude  longitude             timezone
0         Lagos       Nigeria    6.5244     3.3792         Africa/Lagos
1       Nairobi         Kenya   -1.2921    36.8219       Africa/Nairobi
2         Accra         Ghana    5.6037    -0.1870         Africa/Accra
3  Johannesburg  South Africa  -26.2041    28.0473  Africa/Johannesburg
4         Cairo         Egypt   30.0444    31.2357         Africa/Cairo
Fetching Lagos...


2026-10-12 14:19:14  INFO      weather_log  Successfully fetched forecast for lat=6.5244, lon=3.3792, tz=Africa/Lagos
2026-10-12 14:19:14  INFO      weather_log  Successfully fetched data for Lagos
2026-10-12 14:19:14  INFO      weather_log  Fetching Nairobi...


Fetching Nairobi...


2026-10-12 14:19:15  INFO      weather_log  Successfully fetched forecast for lat=-1.2921, lon=36.8219, tz=Africa/Nairobi
2026-10-12 14:19:15  INFO      weather_log  Successfully fetched data for Nairobi
2026-10-12 14:19:15  INFO      weather_log  Fetching Accra...


...

Fetching Johannesburg...


2026-10-12 14:19:17  INFO      weather_log  Successfully fetched forecast for lat=-26.2041, lon=28.0473, tz=Africa/Johannesburg
2026-10-12 14:19:17  INFO      weather_log  Successfully fetched data for Johannesburg
2026-10-12 14:19:17  INFO      weather_log  Fetching Cairo...


Fetching Cairo...


2026-10-12 14:19:18  INFO      weather_log  Successfully fetched forecast for lat=30.0444, lon=31.2357, tz=Africa/Cairo
2026-10-12 14:19:18  INFO      weather_log  Successfully fetched data for Cairo



Combined: (35, 6)
        date  temp_max_c  temp_min_c  rain_mm  wind_max_kmh   city
0 2024-01-01        33.1        24.2      0.0          14.8  Lagos
1 2024-01-02        32.8        23.8      0.0          12.3  Lagos
2 2024-01-03        31.5        24.1      2.4          18.6  Lagos
3 2024-01-04        33.4        24.5      0.0          10.2  Lagos
4 2024-01-05        34.2        25.1      0.0          11.5  Lagos


---

<details>
<summary><strong>Solutions (click to expand)</strong></summary>

### Exercise 1 solution
```python
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude':  -1.2921,
    'longitude': 36.8219,
    'daily':     'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone':  'Africa/Nairobi',
}
response = requests.get(url, params=params)
data = response.json()
print(f'Status: {response.status_code}')
print(f'Timezone: {data["timezone"]}')
print(f'First date: {data["daily"]["time"][0]}')
```

### Exercise 2 solution
```python
def get_forecast(lat, lon, tz, fields='temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max'):
    params = {
        'latitude':  lat,
        'longitude': lon,
        'daily':     fields,
        'timezone':  tz,
    }
    try:
        response = requests.get(
            'https://api.open-meteo.com/v1/forecast',
            params=params,
            timeout=10,
        )
        response.raise_for_status()
        return response.json()
    except requests.RequestException as e:
        print(f'Error: {e}')
        return None

result = get_forecast(5.6037, -0.1870, 'Africa/Accra')
if result:
    print('Got', len(result['daily']['time']), 'days for Accra')
```

### Exercise 3 solution
```python
response = requests.get(bad_url)
print(f'Status: {response.status_code}')
print(response.json())
```

### Exercise 4 solution
```python
data = get_forecast(30.0444, 31.2357, 'Africa/Cairo')
if data:
    out = DATA / 'cairo_forecast.json'
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)
    with open(out, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    print(f'Days: {len(loaded["daily"]["time"])}')
```

### Exercise 5 solution
```python
params = {
    'latitude':  6.5244,
    'longitude': 3.3792,
    'hourly':    'temperature_2m',
    'timezone':  'Africa/Lagos',
}
response = requests.get('https://api.open-meteo.com/v1/forecast', params=params)
data = response.json()
print(f'Hourly values: {len(data["hourly"]["time"])}')  # 168
```

### Exercise 6 solution
```python
with open(DATA / 'sample_weather.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)
df = pd.DataFrame(raw['daily']).rename(columns={
    'time':                'date',
    'temperature_2m_max':  'temp_max_c',
    'temperature_2m_min':  'temp_min_c',
    'precipitation_sum':   'rain_mm',
    'windspeed_10m_max':   'wind_max_kmh',
})
print(df)
```

### Exercise 7 solution
```python
df['temp_range_c'] = df['temp_max_c'] - df['temp_min_c']
df['is_rainy']     = df['rain_mm'] > 0
print(df)
```

### Exercise 8 solution
```python
cities_df = pd.read_csv(DATA / 'cities.csv')
all_frames = []
for _, row in cities_df.iterrows():
    raw = get_forecast(row['latitude'], row['longitude'], row['timezone'])
    if raw:
        df = pd.DataFrame(raw['daily']).rename(columns={
            'time': 'date', 'temperature_2m_max': 'temp_max_c',
            'temperature_2m_min': 'temp_min_c', 'precipitation_sum': 'rain_mm',
            'windspeed_10m_max': 'wind_max_kmh',
        })
        df['city'] = row['city']
        all_frames.append(df)
    time.sleep(0.3)
combined = pd.concat(all_frames, ignore_index=True)
print(combined.shape)
print(combined.head())
```

### Exercise 9 solution
```python
# 1
print(combined.groupby('city')['temp_max_c'].mean().idxmax())
# 2
print(combined.groupby('city')['rain_mm'].sum().idxmax())
# 3
cairo = combined[combined['city'] == 'Cairo']
print((cairo['wind_max_kmh'] > 20).sum())
```

### Exercise 10 solution
```python
def check_weather_data(df):
    ok = True
    if df['temp_max_c'].isnull().any():
        print('ERROR: temp_max_c has null values'); ok = False
    if df['temp_min_c'].isnull().any():
        print('ERROR: temp_min_c has null values'); ok = False
    if not (df['temp_max_c'] > df['temp_min_c']).all():
        print('ERROR: temp_max_c is not always greater than temp_min_c'); ok = False
    if (df['rain_mm'] < 0).any():
        print('ERROR: rain_mm has negative values'); ok = False
    if ok:
        print('Validation passed')
check_weather_data(combined)
```

### Exercise 15 key — upsert pattern
```python
upsert_sql = '''
    INSERT INTO city_weather_ex (city, date, temp_max_c, rain_mm)
    VALUES (%s, %s, %s, %s)
    ON CONFLICT (city, date) DO UPDATE SET
        temp_max_c = EXCLUDED.temp_max_c,
        rain_mm    = EXCLUDED.rain_mm;
'''
```

</details>